모듈 부르고 시스템 형성

In [ ]:
import math
import pychrono as chrono
import pychrono.irrlicht as chronoirr

print("=" * 55)
print("Lesson 06: 재질과 마찰")
print("=" * 55)

sys = chrono.ChSystemNSC()
sys.SetGravitationalAcceleration(chrono.ChVector3d(0, -9.81, 0))
sys.SetCollisionSystemType(chrono.ChCollisionSystem.Type_BULLET)

각 마찰 재질 생성

In [ ]:
# 마찰 계수
#   0.0 = 완전 미끄러움 (얼음 위)
#   0.5 = 보통 (나무 바닥)
#   1.0 = 매우 거침 (고무)

mat_ice = chrono.ChContactMaterialNSC()
mat_ice.SetFriction(0.0)      # 얼음: 전혀 안 멈춤
mat_ice.SetRestitution(0.3)

mat_wood = chrono.ChContactMaterialNSC()
mat_wood.SetFriction(0.5)     # 나무: 보통
mat_wood.SetRestitution(0.3)

mat_rubber = chrono.ChContactMaterialNSC()
mat_rubber.SetFriction(1.0)   # 고무: 바로 멈춤
mat_rubber.SetRestitution(0.3)

materials = [
    (mat_ice,    "얼음 (마찰=0.0)", chrono.ChColor(0.7, 0.85, 0.95)),   # 밝은 파랑
    (mat_wood,   "나무 (마찰=0.5)", chrono.ChColor(0.7, 0.55, 0.3)),    # 갈색
    (mat_rubber, "고무 (마찰=1.0)", chrono.ChColor(0.3, 0.3, 0.3)),     # 어두운 회색
]

print("\n3가지 재질:")
for mat, name, _ in materials:
    print(f"  - {name}")

경사면 3개 생성

In [ ]:
# SetRot() 으로 Z축 기준 20도 회전

ramp_angle = 20  # degree
ramp_rad = math.radians(ramp_angle)  # 라디안으로 변환

print(f"\n경사면 각도: {ramp_angle}°")
print("  같은 각도, 다른 재질의 경사면 3개를 나란히 배치")

ramps = []  # 경사면 객체 리스트 생성
balls = []  # 공 객체 리스트 생성

for i, (mat, name, color) in enumerate(materials):
    z_offset = (i - 1) * 4  # z = -4, 0, 4            # 나란히 배치

    # 경사면 (기울어진 상자)
    ramp = chrono.ChBody()                           # 경사면 객체 생성
    ramp.SetPos(chrono.ChVector3d(-1, 2, z_offset))
    ramp.SetFixed(True)
    ramp.EnableCollision(True)

    # Z축 기준 회전 (경사면 만들기)
    q = chrono.ChQuaterniond()   # 쿼터니언 객체 생성, 3D 회전 표현 방법
    q.SetFromAngleZ(ramp_rad)    # Z축 기준으로 ramp_rad 만큼 회전
    ramp.SetRot(q)               # 만든 회전 적용

    ramp.AddCollisionShape(chrono.ChCollisionShapeBox(mat, 6, 0.3, 2))  # 경사면 보이는 모양 생성
    ramp_shape = chrono.ChVisualShapeBox(6, 0.3, 2)
    ramp_shape.SetColor(color)                           # 색 지정
    ramp.AddVisualShape(ramp_shape)                      
    sys.AddBody(ramp)
    ramps.append(ramp)

    # 경사면 아래에 평평한 바닥 (공이 굴러온 후 마찰 차이를 보기 위해)
    flat = chrono.ChBody()                            # 평평한 바닥 객체 생성
    flat.SetPos(chrono.ChVector3d(5, 0, z_offset))    # 위치 설정
    flat.SetFixed(True)
    flat.EnableCollision(True)
    flat.AddCollisionShape(chrono.ChCollisionShapeBox(mat, 12, 0.3, 2))
    flat_shape = chrono.ChVisualShapeBox(12, 0.3, 2)
    flat_shape.SetColor(chrono.ChColor(
        min(color.R + 0.15, 1.0),  # 색 조금씩 다르게
        min(color.G + 0.15, 1.0),  # 색 조금씩 다르게
        min(color.B + 0.15, 1.0)   # 색 조금씩 다르게
    ))
    flat.AddVisualShape(flat_shape)       # 보이는 바닥 모양 붙이기
    sys.AddBody(flat)

    # 경사면 위에 구 올려놓기
    ball_x = -1 - 2.0 * math.cos(ramp_rad)         # 구의 x위치 계산
    ball_y = 2 + 2.0 * math.sin(ramp_rad) + 0.4    # 구의 y위치 계산, +0.4는 박히지 않기 위함
    ball = chrono.ChBodyEasySphere(0.3, 2000, True, True, mat)
    ball.SetPos(chrono.ChVector3d(ball_x, ball_y, z_offset))         # 구 위치 설정
    ball.GetVisualShape(0).SetColor(chrono.ChColor(0.9, 0.2, 0.2))   # 구 색 지정
    sys.AddBody(ball)
    balls.append(ball)

    print(f"  경사면 {i+1} ({name}): z={z_offset}")

결과 확인

In [ ]:
print("\n반발 계수 비교:")

mat_clay = chrono.ChContactMaterialNSC()    # 찰흙 재질 객체 생성
mat_clay.SetFriction(0.5)
mat_clay.SetRestitution(0.0)                # 탄성 계수 0

mat_normal = chrono.ChContactMaterialNSC()  # 보통 재질 객체 생성
mat_normal.SetFriction(0.5)
mat_normal.SetRestitution(0.5)              # 탄성 계수 0.5

mat_super = chrono.ChContactMaterialNSC()   # 탱탱볼 재질 객체 생성
mat_super.SetFriction(0.5)
mat_super.SetRestitution(0.95)              # 탄성 계수 0.95

bounce_mats = [        # 재질 목록 시작
    (mat_clay,   "찰흙 (반발=0.0)", chrono.ChColor(0.6, 0.4, 0.3)),   # (재질, 이름 색)
    (mat_normal, "보통 (반발=0.5)", chrono.ChColor(0.5, 0.5, 0.5)),
    (mat_super,  "슈퍼볼 (반발=0.95)", chrono.ChColor(0.2, 0.8, 0.3)),
]

bounce_balls = []
for i, (mat, name, color) in enumerate(bounce_mats):
    x_offset = 8 + i * 3  # 오른쪽에 나란히 배치


    pad = chrono.ChBody() # 바닥 객체 생성
    pad.SetPos(chrono.ChVector3d(x_offset, -0.5, 0))
    pad.SetFixed(True)
    pad.EnableCollision(True)
    pad.AddCollisionShape(chrono.ChCollisionShapeBox(mat, 2, 1, 2))
    pad_shape = chrono.ChVisualShapeBox(2, 1, 2) 
    pad_shape.SetColor(color)
    pad.AddVisualShape(pad_shape)
    sys.AddBody(pad)

    bb = chrono.ChBodyEasySphere(0.3, 2000, True, True, mat)      # 바닥이랑 같은 재질 공 생성
    bb.SetPos(chrono.ChVector3d(x_offset, 6, 0))                  # 같은 높이 y=6에서 시작
    bb.GetVisualShape(0).SetColor(chrono.ChColor(0.9, 0.9, 0.2))  # 노랑
    sys.AddBody(bb)
    bounce_balls.append(bb)

    print(f"  {name}: x={x_offset}")

시각화

In [ ]:
print("\n3D 시각화 초기화 중...")

vis = chronoirr.ChVisualSystemIrrlicht()
vis.AttachSystem(sys)
vis.SetWindowSize(1280, 720)
# 참고: macOS Retina에서는 렌더링이 창의 일부만 채울 수 있습니다.
vis.SetWindowTitle('Lesson 06: Friction & Restitution')
vis.Initialize()
vis.AddSkyBox()
vis.AddCamera(chrono.ChVector3d(3, 6, 16), chrono.ChVector3d(3, 1, 0))
vis.AddTypicalLights()

print("시각화 준비 완료!")
print("\n" + "─" * 55)
print("  [왼쪽] 경사면 3개: 마찰 계수 비교")
print("    → 얼음(파랑)은 잘 미끄러지고, 고무(회색)는 안 미끄러짐")
print("  [오른쪽] 바닥 3개: 반발 계수 비교")
print("    → 찰흙은 안 튕기고, 슈퍼볼은 높이 튕김")
print("  마우스로 카메라를 돌려서 양쪽을 관찰하세요!")
print("  ESC → 종료")
print("─" * 55)

시뮬 루프

In [ ]:
step_size = 0.005
realtime_timer = chrono.ChRealtimeStepTimer()

while vis.Run():
    vis.BeginScene()
    vis.Render()
    vis.EndScene()
    sys.DoStepDynamics(step_size)
    realtime_timer.Spin(step_size)

결과 확인

In [ ]:
print(f"\n시뮬레이션 종료! (총 {sys.GetChTime():.2f}초)")

print("\n경사면 실험 결과 (마찰 비교):")
for i, (_, name, _) in enumerate(materials):
    b = balls[i]
    print(f"  {name}: 최종 위치=({b.GetPos().x:.2f}, {b.GetPos().y:.2f})")

print("\n튕김 실험 결과 (반발 비교):")
for i, (_, name, _) in enumerate(bounce_mats):
    b = bounce_balls[i]
    print(f"  {name}: 최종 높이={b.GetPos().y:.3f} m")

print(f"""
핵심 정리:
  1. 마찰 계수 ↑ → 물체가 경사면에서 덜 미끄러짐
  2. 반발 계수 ↑ → 물체가 더 높이 튕김
  3. 같은 형태/질량이라도 재질에 따라 움직임이 완전히 달라짐
  4. SetRot(quaternion) 으로 물체를 기울일 수 있음

다음 단계:
  → lesson_07에서 회전 조인트(경첩)를 배웁니다
""")
